# Accuracy Comparison: CIFAR-10 (Float vs. KAN vs. Integer-KAN)

In [ ]:
import os
import sys
import importlib
import inspect

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import converted_KAN
import converted_KAN.accuracy_cost as accuracy_cost

importlib.reload(accuracy_cost)

from converted_KAN import (
    convert_to_kan,
    convert_to_int_kan,
    IntKANWrapper,
)
from converted_KAN.ops_counter import count_ops

evaluate_accuracy = accuracy_cost.evaluate_accuracy

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
def eval_with_progress(model, loader, device):
    sig = inspect.signature(evaluate_accuracy)
    if "show_progress" in sig.parameters:
        return evaluate_accuracy(model, loader, device=device, show_progress=True)
    return evaluate_accuracy(model, loader, device=device)

## Load model (CIFAR10 ResNet20 from torch.hub)

In [ ]:
HUB_REPO = "chenyaofo/pytorch-cifar-models"
HUB_MODEL = "cifar10_resnet20"

model = torch.hub.load(HUB_REPO, HUB_MODEL, pretrained=True, verbose=True)
model = model.to(device).eval()
model

## CIFAR-10 test set

In [ ]:
BATCH_SIZE = 128
MAX_SAMPLES = 2000  # set to None for full test set

normalize = transforms.Normalize(
    mean=[0.4914, 0.4822, 0.4465],
    std=[0.2470, 0.2435, 0.2616],
)

transform = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])

test_dataset = datasets.CIFAR10(root="data", train=False, download=True, transform=transform)
if MAX_SAMPLES is not None:
    test_dataset = Subset(test_dataset, list(range(MAX_SAMPLES)))

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
len(test_loader)

## Accuracy Comparison

In [ ]:
# 1) Float model
acc_float = eval_with_progress(model, test_loader, device)

# 2) KAN (Float)
kan_model = convert_to_kan(model, inplace=False).to(device).eval()
acc_kan = eval_with_progress(kan_model, test_loader, device)

# 3) Integer-KAN (fixed-point), all-int path
frac_bits = 8
int_model = convert_to_int_kan(model, frac_bits=frac_bits, int_conv=True, int_relu=True)
int_runner = IntKANWrapper(
    int_model,
    frac_bits=frac_bits,
    return_int=True,
    quantize_input=True,
).to(device).eval()
acc_int_kan = eval_with_progress(int_runner, test_loader, device)

print(f"Accuracy Float:     {acc_float:.4f}")
print(f"Accuracy KAN:       {acc_kan:.4f}")
print(f"Accuracy Int-KAN:   {acc_int_kan:.4f}")

## Attention Comparison (AttentionKAN vs. nn.MultiheadAttention)

In [ ]:
import torch.nn as nn
from converted_KAN.attentionkan import AttentionKAN

embed_dim = 64
num_heads = 4
seq_len = 16
batch_size = 2

attn_kan = AttentionKAN(embed_dim, num_heads, dropout=0.0).to(device).eval()
attn_std = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.0, batch_first=True).to(device).eval()

with torch.no_grad():
    attn_std.in_proj_weight.copy_(torch.cat([
        attn_kan.q_proj.weight,
        attn_kan.k_proj.weight,
        attn_kan.v_proj.weight,
    ], dim=0))
    attn_std.in_proj_bias.copy_(torch.cat([
        attn_kan.q_proj.bias,
        attn_kan.k_proj.bias,
        attn_kan.v_proj.bias,
    ], dim=0))
    attn_std.out_proj.weight.copy_(attn_kan.out_proj.weight)
    attn_std.out_proj.bias.copy_(attn_kan.out_proj.bias)

x = torch.randn(batch_size, seq_len, embed_dim, device=device)

with torch.no_grad():
    result_kan = attn_kan(x, x, x)
    result_std, _ = attn_std(x, x, x)

diff = (result_std - result_kan).abs()
print("Attention comparison:")
print(f"  Output shape: {result_kan.shape}")
print(f"  Max difference: {diff.max().item():.2e}")
print(f"  Mean difference: {diff.mean().item():.2e}")
print(f"  Numerically equal (atol=1e-4): {torch.allclose(result_std, result_kan, atol=1e-4)}")

## Ops Counter: KAN vs. non-KAN (including per-layer)

In [ ]:
input_shape = (1, 3, 32, 32)

base_counts = count_ops(model, input_shape, device=device, per_layer=True)
kan_counts = count_ops(kan_model, input_shape, device=device, per_layer=True)

print("Non-KAN total:\n", base_counts["total"])
print("\nKAN total:\n", kan_counts["total"])

print("\nNon-KAN per-layer:\n", base_counts["per_layer"])
print("\nKAN per-layer:\n", kan_counts["per_layer"])